# 14 — FL scale-up: does delta-space federation generalize beyond C_110?

Notebook 11 proved the pattern on one specialty: level-FedAvg breaks on non-IID scale, delta-space FedAvg
recovers pooled performance. The scale-up question is whether that holds across **all 24 specialties** and
the full public `fl_client` pool — same model (change = a + b·momentum), same declared splits and regimes,
nothing tuned per specialty.

Protocol, unchanged from 11:

- **Clients** = public `fl_client` providers, one cohort per specialty (each provider is a client in every
  specialty it reports).
- **Splits**: train origins before Apr 2024 (full-history vs post-Covid from Mar 2020), validation FY 2024-25,
  sealed FY 2025-26 untouched. **Persistence sits in every row of the results table.**
- **The C_110 row must reproduce notebook 11 exactly** — asserted below (1,407 rows, naive 153, fedavg-delta
  154.0, one cold-start client). Any drift in cohort or split trips it.
- R² was proposed externally and stays rejected (dominated by cross-provider size variance — see 11's triage).
- Results go to `data/processed/` like every other modelling notebook — notebook 13 remains the only writer
  of `output/` artifacts (deviation from the pasted spec, documented here).

In [1]:
import pandas as pd                          # tables
import numpy as np                           # lstsq fits and arithmetic guards
from pathlib import Path

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25 (same as 04/11)
COVID = pd.Timestamp("2020-03-01")           # declared regime break (unchanged)

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[(panel["month"] < TEST_START) & panel["fl_client"] & panel["is_public"]]
print(dev["tfunc_code"].nunique(), "specialties,", dev["provider_code"].nunique(),
      "public fl_client providers,", f"{len(dev):,}", "provider-months in the scale-up pool")

24 specialties, 149 public fl_client providers, 187,041 provider-months in the scale-up pool


## One loop, every specialty

Per specialty: build targets and momentum per provider, split by the declared dates, fit four delta models
(local per client, pooled, fedavg-delta, fedavg-delta with client-kept intercept), score everything on that
specialty's FY 2024-25 rows against persistence. Cold starts — providers whose series in this specialty
begins inside the validation year, the RJ8 case generalized — get neutral zero momentum and are counted per
specialty; clients with no training rows fall back to persistence locally while the shared model can serve them.

In [2]:
# Goal: the per-specialty results table - every model, both regimes, MAE and RMSE.
def fit_delta(rows):                         # change = a + b * last month's change (11 phase 2 model)
    X = np.column_stack([np.ones(len(rows)), rows["delta_1"].to_numpy()])
    beta, *_ = np.linalg.lstsq(X, (rows["target"] - rows["total_waiting"]).to_numpy(), rcond=None)
    return beta

results = []
for tf, df in dev.groupby("tfunc_code"):
    df = df.sort_values(["provider_code", "month"]).copy()
    g = df.groupby("provider_code")["total_waiting"]
    df["target"] = g.shift(-1)               # next month's list
    df["delta_1"] = df["total_waiting"] - g.shift(1)   # 1-month momentum
    df = df.dropna(subset=["target"])

    train_full = df[df["month"] < VAL_START].dropna(subset=["delta_1"])   # first row per provider
    train_post = train_full[train_full["month"] >= COVID]                 # has no momentum yet
    val = df[df["month"] >= VAL_START].copy()
    assert len(val) == 0 or val["month"].max() < TEST_START
    # the sealed-year sentinel every modelling notebook carries (04/07/08/11); a sealed-year origin
    # would trip it. The len == 0 branch is X01, which has no validation rows in this pool.
    n_cold = int(val["delta_1"].isna().sum())          # series starting inside FY 2024-25
    val["delta_1"] = np.where(val["delta_1"].notna(), val["delta_1"], 0.0)  # zero change = neutral

    truth = val["target"].to_numpy()
    current = val["total_waiting"].to_numpy()
    mom = val["delta_1"].to_numpy()
    def score(pred):                          # MAE, RMSE on this specialty's validation rows
        err = truth - pred
        return float(np.abs(err).mean()), float(np.sqrt((err ** 2).mean()))

    row = {"tfunc_code": tf, "n_val": len(val), "n_clients": val["provider_code"].nunique(),
           "n_cold_start": n_cold}
    row["persistence_mae"], row["persistence_rmse"] = score(current)
    for regime, tr in [("full", train_full), ("post", train_post)]:
        bd = {p: (fit_delta(r), len(r)) for p, r in tr.groupby("provider_code")}
        n_tot = sum(n for _, n in bd.values())
        bfed = sum(b * n for b, n in bd.values()) / n_tot   # FedAvg: sample-weighted coefficient average
        bpool = fit_delta(tr)

        a = val["provider_code"].map({p: b[0] for p, (b, n) in bd.items()})       # local intercepts
        slope = val["provider_code"].map({p: b[1] for p, (b, n) in bd.items()})   # local slopes
        has_local = a.notna().to_numpy()      # clients with zero training rows have no local model
        local = np.where(has_local, current + a.fillna(0).to_numpy() + slope.fillna(0).to_numpy() * mom,
                         current)             # ...and fall back to persistence
        pooled = current + bpool[0] + bpool[1] * mom
        fed = current + bfed[0] + bfed[1] * mom
        fed_local_a = current + np.where(has_local, a.fillna(0).to_numpy(), 0.0) + bfed[1] * mom

        for model, pred in [("local", local), ("pooled", pooled),
                            ("fedavg_delta", fed), ("fedavg_delta_local_a", fed_local_a)]:
            row[f"{model}_{regime}_mae"], row[f"{model}_{regime}_rmse"] = score(pred)
    results.append(row)

res = pd.DataFrame(results)
c110 = res.set_index("tfunc_code").loc["C_110"]
assert int(c110["n_val"]) == 1407 and int(c110["n_cold_start"]) == 1        # 11's rows and RJ8
assert round(c110["persistence_mae"]) == 153                                # the standing anchor
assert round(c110["fedavg_delta_full_mae"], 1) == 154.0, c110["fedavg_delta_full_mae"]
# these three fire if this loop's cohort or splits drift from what notebook 11 scored
print(len(res), "specialties scored;", int(res["n_val"].sum()), "validation rows in total")

C:\Users\heave\AppData\Local\Temp\ipykernel_26204\3314266522.py:29: RuntimeWarning: Mean of empty slice
  return float(np.abs(err).mean()), float(np.sqrt((err ** 2).mean()))
C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


24 specialties scored; 26504 validation rows in total


In [3]:
# Goal: the readable verdict table + stored csv.
res = res.sort_values("n_val", ascending=False)
res.to_csv(PROC / "fl_scaleup_by_specialty.csv", index=False)

cols = ["tfunc_code", "n_val", "n_clients", "n_cold_start", "persistence_mae",
        "pooled_full_mae", "fedavg_delta_full_mae", "local_full_mae"]
print(res[cols].round(1).to_string(index=False))

gap_pooled = res["fedavg_delta_full_mae"] - res["pooled_full_mae"]
beats_naive = int((res["fedavg_delta_full_mae"] < res["persistence_mae"]).sum())
print(f"\nfedavg-delta beats persistence in {beats_naive} of {len(res)} specialties (full regime)")
print(f"fedavg-delta vs pooled MAE gap (full): median {gap_pooled.median():+.2f}, "
      f"worst {gap_pooled.max():+.2f}, best {gap_pooled.min():+.2f}")
print(f"cold-start provider-rows across all specialties: {int(res['n_cold_start'].sum())}")
print(res.shape, "->", PROC / "fl_scaleup_by_specialty.csv")

tfunc_code  n_val  n_clients  n_cold_start  persistence_mae  pooled_full_mae  fedavg_delta_full_mae  local_full_mae
       X02   1409        129             1            144.2            152.6                  153.2           158.9
     C_110   1407        128             1            153.5            153.7                  154.0           154.5
     C_320   1349        123             0            117.3            116.6                  116.3           117.3
       X06   1331        125             0             89.9             90.6                   91.8            94.9
     C_340   1328        122             1             82.2             82.0                   81.9            83.7
     C_410   1328        121             0             52.1             51.3                   51.3            53.1
       X04   1323        122             0             92.4            100.0                   98.7           107.3
     C_502   1320        120             0            131.9            1